**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Information Theory

One framework that explains compression limits, communication limits, and — through KL divergence — why cross-entropy is *the* machine learning loss. Four sessions from 'what is a bit?' to the channel coding theorem, with every quantity computed live.

## 0. Introduction

Shannon's move: measure information as **surprise**. Rare events carry more information than expected ones, and $-\log_2 p$ is the unique surprise measure (up to base) that is continuous, decreasing in $p$, and additive over independent events.

## 1. Pre-requisites

[Random Variables](../Analysis/Random_Variables.ipynb) (expectation, distributions); [Independence](../Analysis/Independence.ipynb) for the coding arguments.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

def H(p):
    """Entropy in bits of a probability vector."""
    p = np.asarray(p, float); p = p[p > 0]
    return -(p * np.log2(p)).sum()

---
### 🕐 Session 1 of 4 — *Entropy* (~35 min)
**Goal:** quantify average surprise; see entropy as the compression limit.
**Builds on:** [Random Variables](../Analysis/Random_Variables.ipynb). &nbsp; **Feeds into:** Session 2 (KL & cross-entropy).

---

## 2. Entropy

💡 **Intuition.** Entropy $H(X) = E[-\log_2 p(X)]$ is the **average surprise** of a source — equivalently, the number of yes/no questions you need *on average* to pin down an outcome, when you ask cleverly. A fair coin: 1 bit. A loaded coin: less, because you can exploit the bias. That question-count reading *is* the compression story: you cannot losslessly encode a source below $H$ bits/symbol on average (Shannon's source coding theorem), and Huffman codes get within 1 bit of it.

In [2]:
p = np.linspace(0.001, 0.999, 500)
plt.figure(figsize=(7, 2.6))
plt.plot(p, [H([q, 1-q]) for q in p])
plt.xlabel("P(heads)"); plt.ylabel("H [bits]")
plt.title("Binary entropy: maximal at fair (1 bit), zero when certain")
plt.grid(True); plt.tight_layout(); plt.show()

/tmp/ipykernel_2018428/3520344432.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(True); plt.tight_layout(); plt.show()


In [3]:
# Entropy = compression limit, demonstrated with a real compressor (zlib)
import zlib
for p1 in [0.5, 0.9, 0.99]:
    bits = (rng.random(200_000) < p1).astype(np.uint8)
    packed = np.packbits(bits).tobytes()
    comp = zlib.compress(packed, 9)
    rate = 8 * len(comp) / len(bits)
    print(f"P(1)={p1:4}: entropy {H([p1, 1-p1]):.3f} bits/sym   zlib achieved {rate:.3f} bits/sym")
print("→ a general-purpose compressor hugs the entropy floor it can never beat")

P(1)= 0.5: entropy 1.000 bits/sym   zlib achieved 1.001 bits/sym
P(1)= 0.9: entropy 0.469 bits/sym   zlib achieved 0.565 bits/sym
P(1)=0.99: entropy 0.081 bits/sym   zlib achieved 0.115 bits/sym
→ a general-purpose compressor hugs the entropy floor it can never beat


---
### 🕐 Session 2 of 4 — *KL Divergence & Cross-Entropy* (~40 min)
**Goal:** measure the cost of believing the wrong distribution; derive the ML loss.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (mutual information).

---

## 3. Relative Entropy

💡 **Intuition.** Suppose the world emits symbols from $p$ but you built your code (or your model) for $q$. You pay $E_p[-\log_2 q(X)]$ bits per symbol — the **cross-entropy** — instead of the optimal $H(p)$. The overpayment is the KL divergence:
$D(p\|q) = \sum_x p(x) \log \frac{p(x)}{q(x)} \ge 0$ — *the price of wrong beliefs, in bits*. Minimizing cross-entropy in ML is minimizing that price: training a classifier literally optimizes its codebook for the data distribution.

### Proof: $D(p\|q) \ge 0$ (Gibbs' inequality)

Since $\log$ is concave, Jensen's inequality gives
$$-D(p\|q) = \sum_x p(x) \log\frac{q(x)}{p(x)} \le \log \sum_x p(x)\frac{q(x)}{p(x)} = \log \sum_x q(x) = \log 1 = 0,$$
with equality iff $p = q$. $\blacksquare$ Two warnings: $D$ is **not symmetric** and violates the triangle inequality — a directed cost, not a distance.

In [4]:
# Cross-entropy loss IS log-loss with a KL floor
p_true = np.array([0.7, 0.2, 0.1])
qs = {"perfect  q=p": p_true, "close": np.array([0.6, 0.3, 0.1]), "wrong": np.array([0.1, 0.2, 0.7])}
for name, q in qs.items():
    ce = -(p_true * np.log2(q)).sum()
    print(f"{name:14s} cross-entropy {ce:.3f} bits = H(p) {H(p_true):.3f} + KL {ce - H(p_true):.3f}")

perfect  q=p   cross-entropy 1.157 bits = H(p) 1.157 + KL 0.000
close          cross-entropy 1.195 bits = H(p) 1.157 + KL 0.039
wrong          cross-entropy 2.841 bits = H(p) 1.157 + KL 1.684


---
### 🕐 Session 3 of 4 — *Mutual Information* (~35 min)
**Goal:** quantify what one variable tells you about another; data processing can only lose it.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (coding at a glance).

---

## 4. Mutual Information

💡 **Intuition.** $I(X;Y) = H(X) - H(X|Y)$: how many bits of uncertainty about $X$ does observing $Y$ remove? Equivalently $I(X;Y) = D(p_{XY} \| p_X p_Y)$ — the KL cost of pretending they're independent. Zero iff independent ([Independence](../Analysis/Independence.ipynb), quantified!), and — unlike correlation — it detects *nonlinear* dependence too.

In [5]:
def mi_hist(x, y, bins=24):
    pxy, _, _ = np.histogram2d(x, y, bins=bins, density=False)
    pxy = pxy / pxy.sum()
    px, py = pxy.sum(1, keepdims=True), pxy.sum(0, keepdims=True)
    mask = pxy > 0
    return (pxy[mask] * np.log2(pxy[mask] / (px @ py)[mask])).sum()

N = 100_000
x = rng.standard_normal(N)
pairs = {"independent": rng.standard_normal(N),
         "linear y=x+n": x + 0.5 * rng.standard_normal(N),
         "NONLINEAR y=x²+n": x**2 + 0.5 * rng.standard_normal(N)}
for name, y in pairs.items():
    r = np.corrcoef(x, y)[0, 1]
    print(f"{name:18s} correlation {r:+.3f}   mutual information {mi_hist(x, y):.3f} bits")
print("→ correlation misses y=x²; mutual information doesn't")

independent        correlation +0.000   mutual information 0.003 bits
linear y=x+n       correlation +0.894   mutual information 1.106 bits
NONLINEAR y=x²+n   correlation -0.001   mutual information 0.979 bits
→ correlation misses y=x²; mutual information doesn't


**Data processing inequality** (stated): if $X \to Y \to Z$ is a Markov chain (Z computed from Y alone), then $I(X;Z) \le I(X;Y)$ — **no processing can create information about $X$**. Deep networks, filters, features: every stage can only preserve or destroy. This is the information-theoretic backbone of representation learning, and the reason 'garbage in, garbage out' is a theorem.

---
### 🕐 Session 4 of 4 — *Coding at a Glance* (~35 min)
**Goal:** the two Shannon theorems, one channel capacity computed, and the SNR connection.
**Builds on:** Sessions 1–3.

---

## 5. The Two Theorems

**Source coding** (S1's floor): lossless compression needs $\ge H$ bits/symbol.

**Channel coding.** A noisy channel has capacity $C = \max_{p_X} I(X;Y)$; *any* rate below $C$ is achievable with vanishing error (via long random-ish codes), and no rate above it is. The shocking part: noise does **not** cap reliability — only *rate*.

💡 **Intuition.** Long codewords let the law of large numbers ([Independence](../Analysis/Independence.ipynb)) concentrate the noise: typical received sequences cluster into disjoint balls around codewords, and decoding is 'which ball am I in?'. Capacity counts how many disjoint balls fit.

In [6]:
# Capacity of the binary symmetric channel: C = 1 − H(ε)
eps = np.linspace(0, 0.5, 200)
C = 1 - np.array([H([e, 1-e]) if 0 < e else 0 for e in eps])
plt.figure(figsize=(7, 2.6))
plt.plot(eps, C)
plt.xlabel("bit-flip probability ε"); plt.ylabel("capacity [bits/use]")
plt.title("BSC capacity: 1 bit when clean, 0 at ε = ½ (pure noise)")
plt.grid(True); plt.tight_layout(); plt.show()

/tmp/ipykernel_2018428/1934163155.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(True); plt.tight_layout(); plt.show()


**The formula on every comms slide.** For the Gaussian channel with signal-to-noise ratio SNR: $C = \tfrac12 \log_2(1 + \mathrm{SNR})$ bits per use — bandwidth and SNR are the currency of every link budget in [Digital Communications](../../Intro_DSP/Digital_Communications.ipynb).

## 6. Conclusion

Entropy = surprise = compression floor; KL = the bits you waste believing $q$ when truth is $p$ (cross-entropy loss, demystified); mutual information = dependence in bits, immune to nonlinearity, only ever destroyed by processing; capacity = the rate ceiling noise imposes. Four numbers that govern every pipeline in this curriculum.

---
## Where next

- [Digital Communications](../../Intro_DSP/Digital_Communications.ipynb) — engineering toward capacity.
- [Training Dynamics](../../Intro_Mach_Learn/Training_Dynamics.ipynb) — cross-entropy at work, at scale.
- [Estimation Theory](../Estimation_Theory/Estimation_Theory.ipynb) — Fisher information: KL's local curvature.